# Notebook 05 – Gold Layer Validation

## Objective
Validate the Gold layer Delta tables to ensure structural integrity, primary key uniqueness, referential integrity, and financial metric consistency prior to building the Power BI semantic model.

## Pipeline Position
* **Upstream Dependency:** Consumes Gold Delta tables produced in `Notebook 04 – Build Gold Analytical Model`.
* **Downstream Target:** Serves as the final data quality check before Power BI semantic model ingestion and dashboard reporting.

## Scope & Gold Tables
Validates 7 Gold layer Delta tables:
* **Dimensions:** `Calendar`, `products`, `customers`, `regions`
* **Facts:** `Sales`, `marketing_campaign`, `salestarget`

| Catalog Table Name | Entity Type | Business Key / Primary Key | Key Grain / Description |
| :--- | :--- | :--- | :--- |
| `Calendar` | Dimension | `Date` | 1 record per calendar day |
| `products` | Dimension | `ProductID` | 1 record per product |
| `customers` | Dimension | `CustomerID` | 1 record per customer |
| `regions` | Dimension | `Region` | 1 record per region |
| `Sales` | Fact | `OrderLineID` | Foreign keys to `Calendar`, `products`, `customers`, `regions` |
| `marketing_campaign` | Fact | `CampaignID` | Independent marketing metrics |
| `salestarget` | Fact | `TargetID` | Target grain: `Year` + `Month` + `Region` + `ProductCategory` |

## Validation Areas
1. **Table Availability & Row Counts:** Confirm all 7 Gold tables exist and contain records.
2. **Key Integrity:** Ensure zero nulls and zero duplicate values in primary keys.
3. **Referential Integrity:** Ensure zero orphan records exist between `Sales` and dimension tables.
4. **Grain Uniqueness:** Check `salestarget` for duplicate target entries.
5. **Business Logic & Metrics:** Validate non-negative metrics, discount boundaries, and line amount calculations.
6. **Cross-Fact Reconciliation:** Evaluate actual sales against targets and compare marketing spend/revenue against sales totals.

## Expected Outcome
Verify Gold layer readiness for semantic modeling and document intentional synthetic data characteristics.


## 1. Table Availability & Record Inventory

Confirms that all 7 Gold Delta tables are registered in the catalog and reports record counts for baseline verification.

In [3]:
gold_tables = [
    "Calendar",
    "products",
    "customers",
    "regions",
    "Sales",
    "marketing_campaign",
    "SalesTarget"
]

for table_name in gold_tables:
    try:
        df = spark.table(table_name)
        print(f"✓ {table_name}: {df.count()} rows")
    except Exception as e:
        print(f"✗ {table_name}: NOT AVAILABLE")

StatementMeta(, f79bd707-cd04-4b13-bd27-4f59f415bad0, 5, Finished, Available, Finished, False)

✓ Calendar: 348 rows
✓ products: 50 rows
✓ customers: 1000 rows
✓ regions: 4 rows
✓ Sales: 2471 rows
✓ marketing_campaign: 300 rows
✓ SalesTarget: 221 rows


## 2. Primary Key Integrity

Checks all Gold tables for primary key completeness (zero nulls) and uniqueness (zero duplicate keys).

In [7]:
from pyspark.sql.functions import col

key_checks = {
    "Calendar": "Date",
    "products": "ProductID",
    "customers": "CustomerID",
    "regions": "Region",
    "Sales": "OrderLineID",
    "marketing_campaign": "CampaignID",
    "salestarget": "TargetID"
}

print("=== GOLD KEY INTEGRITY CHECK ===\n")

for table_name, key in key_checks.items():

    df = spark.table(table_name)

    null_count = (
        df.filter(col(key).isNull())
        .count()
    )

    duplicate_count = (
        df.groupBy(key)
        .count()
        .filter(col("count") > 1)
        .count()
    )

    print(
        f"{table_name:<20} "
        f"Null {key}: {null_count:<5} "
        f"Duplicate {key}: {duplicate_count}"
    )

StatementMeta(, f79bd707-cd04-4b13-bd27-4f59f415bad0, 9, Finished, Available, Finished, False)

=== GOLD KEY INTEGRITY CHECK ===

Calendar             Null Date: 0     Duplicate Date: 0
products             Null ProductID: 0     Duplicate ProductID: 0
customers            Null CustomerID: 0     Duplicate CustomerID: 0
regions              Null Region: 0     Duplicate Region: 0
Sales                Null OrderLineID: 0     Duplicate OrderLineID: 0
marketing_campaign   Null CampaignID: 0     Duplicate CampaignID: 0
salestarget          Null TargetID: 0     Duplicate TargetID: 0


## 3. Sales Referential Integrity

Validates foreign key references between `Sales` and dimension tables (`products`, `customers`, `regions`, `Calendar`) using anti-joins to detect orphan records.

In [8]:
from pyspark.sql.functions import col

sales = spark.table("Sales")
calendar = spark.table("Calendar")
product = spark.table("products")
customer = spark.table("customers")
region = spark.table("regions")

print("=== SALES REFERENTIAL INTEGRITY ===\n")

# Sales → Product
missing_products = (
    sales.select("ProductID")
    .distinct()
    .join(
        product.select("ProductID").distinct(),
        "ProductID",
        "left_anti"
    )
    .count()
)

# Sales → Customer
missing_customers = (
    sales.select("CustomerID")
    .distinct()
    .join(
        customer.select("CustomerID").distinct(),
        "CustomerID",
        "left_anti"
    )
    .count()
)

# Sales → Region
missing_regions = (
    sales.select("Region")
    .distinct()
    .join(
        region.select("Region").distinct(),
        "Region",
        "left_anti"
    )
    .count()
)

# Sales → Calendar
sales_dates = (
    sales
    .select(col("OrderDate").cast("date").alias("Date"))
    .distinct()
)

calendar_dates = (
    calendar
    .select(col("Date").cast("date").alias("Date"))
    .distinct()
)

missing_dates = (
    sales_dates
    .join(calendar_dates, "Date", "left_anti")
    .count()
)

print("Sales → Product missing keys :", missing_products)
print("Sales → Customer missing keys:", missing_customers)
print("Sales → Region missing keys  :", missing_regions)
print("Sales → Calendar missing keys:", missing_dates)

StatementMeta(, f79bd707-cd04-4b13-bd27-4f59f415bad0, 10, Finished, Available, Finished, False)

=== SALES REFERENTIAL INTEGRITY ===

Sales → Product missing keys : 0
Sales → Customer missing keys: 0
Sales → Region missing keys  : 0
Sales → Calendar missing keys: 0


## 4. SalesTarget Grain & Duplicate Analysis

Evaluates whether `salestarget` maintains a unique 1:1 grain at the `Year` + `Month` + `Region` + `ProductCategory` level.

In [9]:
targets = spark.table("salestarget")

targets.groupBy(
    "Year",
    "Month",
    "Region",
    "ProductCategory"
).count().filter(
    col("count") > 1
).show()

StatementMeta(, f79bd707-cd04-4b13-bd27-4f59f415bad0, 11, Finished, Available, Finished, False)

+----+-----+------+---------------+-----+
|Year|Month|Region|ProductCategory|count|
+----+-----+------+---------------+-----+
|2025|    2| North|      Drinkware|    2|
|2025|    1| North|         Office|    2|
+----+-----+------+---------------+-----+



In [1]:
from pyspark.sql.functions import col

duplicates = (
    spark.table("salestarget")
    .groupBy(
        "Year",
        "Month",
        "Region",
        "ProductCategory"
    )
    .count()
    .filter(col("count") > 1)
)

display(
    spark.table("salestarget")
    .join(
        duplicates.select(
            "Year",
            "Month",
            "Region",
            "ProductCategory"
        ),
        ["Year", "Month", "Region", "ProductCategory"],
        "inner"
    )
    .orderBy(
        "Year",
        "Month",
        "Region",
        "ProductCategory"
    )
)

StatementMeta(, 13aafc3d-a581-45f3-929e-01d73f833544, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4af60556-f99e-4547-b52c-98e97cecb305)

## 5. Sales Business Logic Validation

Validates numeric ranges (non-negative price, cost, quantity), discount boundaries (0%–100%), and mathematical precision of `LineAmount` calculations.

In [2]:
from pyspark.sql.functions import (
    col,
    sum as spark_sum,
    count,
    when,
    abs as spark_abs
)

sales = spark.table("Sales")

# Convert numeric columns
sales_check = (
    sales
    .withColumn("Quantity", col("Quantity").cast("double"))
    .withColumn("UnitPrice", col("UnitPrice").cast("double"))
    .withColumn("CostAtSale", col("CostAtSale").cast("double"))
    .withColumn("Discount", col("Discount").cast("double"))
    .withColumn("LineAmount", col("LineAmount").cast("double"))
)

# Expected revenue after discount
sales_check = sales_check.withColumn(
    "ExpectedLineAmount",
    col("Quantity") *
    col("UnitPrice") *
    (1 - col("Discount"))
)

# Validation flags
sales_check = (
    sales_check
    .withColumn(
        "InvalidQuantity",
        when(col("Quantity") <= 0, 1).otherwise(0)
    )
    .withColumn(
        "InvalidUnitPrice",
        when(col("UnitPrice") < 0, 1).otherwise(0)
    )
    .withColumn(
        "InvalidCost",
        when(col("CostAtSale") < 0, 1).otherwise(0)
    )
    .withColumn(
        "InvalidDiscount",
        when(
            (col("Discount") < 0) | (col("Discount") > 1),
            1
        ).otherwise(0)
    )
    .withColumn(
        "AmountMismatch",
        when(
            spark_abs(
                col("LineAmount") - col("ExpectedLineAmount")
            ) > 0.01,
            1
        ).otherwise(0)
    )
)

print("=== SALES BUSINESS VALIDATION ===")

print(
    "Invalid Quantity:",
    sales_check.filter(col("InvalidQuantity") == 1).count()
)

print(
    "Invalid Unit Price:",
    sales_check.filter(col("InvalidUnitPrice") == 1).count()
)

print(
    "Invalid Cost:",
    sales_check.filter(col("InvalidCost") == 1).count()
)

print(
    "Invalid Discount:",
    sales_check.filter(col("InvalidDiscount") == 1).count()
)

print(
    "Line Amount Mismatch:",
    sales_check.filter(col("AmountMismatch") == 1).count()
)

# Overall financial summary
sales_check.select(
    spark_sum("Quantity").alias("TotalQuantity"),
    spark_sum("LineAmount").alias("TotalRevenue"),
    spark_sum(
        col("Quantity") * col("CostAtSale")
    ).alias("TotalCost")
).show()

StatementMeta(, 13aafc3d-a581-45f3-929e-01d73f833544, 4, Finished, Available, Finished, False)

=== SALES BUSINESS VALIDATION ===
Invalid Quantity: 0
Invalid Unit Price: 0
Invalid Cost: 0
Invalid Discount: 0
Line Amount Mismatch: 0
+-------------+-----------------+-----------------+
|TotalQuantity|     TotalRevenue|        TotalCost|
+-------------+-----------------+-----------------+
|       7446.0|451271.9299999988|357210.3599999999|
+-------------+-----------------+-----------------+



## 6. Marketing Campaign Validation

Checks marketing metrics for valid non-negative values and confirms campaign funnel logic (`Clicks` $\le$ `Impressions`, `Conversions` $\le$ `Clicks`).

In [1]:
from pyspark.sql.functions import col, sum as spark_sum, when

campaigns = spark.table("marketing_campaign")

campaigns_check = (
    campaigns
    .withColumn("Spend", col("Spend").cast("double"))
    .withColumn("RevenueGenerated", col("RevenueGenerated").cast("double"))
    .withColumn("Impressions", col("Impressions").cast("double"))
    .withColumn("Clicks", col("Clicks").cast("double"))
    .withColumn("Conversions", col("Conversions").cast("double"))
)

# Business validation flags
campaigns_check = (
    campaigns_check
    .withColumn(
        "InvalidSpend",
        when(col("Spend") < 0, 1).otherwise(0)
    )
    .withColumn(
        "InvalidRevenue",
        when(col("RevenueGenerated") < 0, 1).otherwise(0)
    )
    .withColumn(
        "InvalidImpressions",
        when(col("Impressions") <= 0, 1).otherwise(0)
    )
    .withColumn(
        "InvalidClicks",
        when(
            (col("Clicks") < 0) |
            (col("Clicks") > col("Impressions")),
            1
        ).otherwise(0)
    )
    .withColumn(
        "InvalidConversions",
        when(
            (col("Conversions") < 0) |
            (col("Conversions") > col("Clicks")),
            1
        ).otherwise(0)
    )
)

print("=== MARKETING CAMPAIGN BUSINESS VALIDATION ===")

print(
    "Invalid Spend:",
    campaigns_check.filter(col("InvalidSpend") == 1).count()
)

print(
    "Invalid Revenue:",
    campaigns_check.filter(col("InvalidRevenue") == 1).count()
)

print(
    "Invalid Impressions:",
    campaigns_check.filter(col("InvalidImpressions") == 1).count()
)

print(
    "Invalid Clicks:",
    campaigns_check.filter(col("InvalidClicks") == 1).count()
)

print(
    "Invalid Conversions:",
    campaigns_check.filter(col("InvalidConversions") == 1).count()
)

# Overall campaign metrics
campaigns_check.select(
    spark_sum("Spend").alias("TotalSpend"),
    spark_sum("RevenueGenerated").alias("TotalCampaignRevenue"),
    spark_sum("Impressions").alias("TotalImpressions"),
    spark_sum("Clicks").alias("TotalClicks"),
    spark_sum("Conversions").alias("TotalConversions")
).show()

StatementMeta(, ee3da146-c5f9-430a-a980-1cf85008508b, 3, Finished, Available, Finished, False)

=== MARKETING CAMPAIGN BUSINESS VALIDATION ===
Invalid Spend: 0
Invalid Revenue: 0
Invalid Impressions: 0
Invalid Clicks: 0
Invalid Conversions: 0
+-----------------+--------------------+----------------+-----------+----------------+
|       TotalSpend|TotalCampaignRevenue|TotalImpressions|TotalClicks|TotalConversions|
+-----------------+--------------------+----------------+-----------+----------------+
|474706.2899999998|  1309804.4799999995|       8359562.0|   740246.0|         61441.0|
+-----------------+--------------------+----------------+-----------+----------------+



## 7. Sales vs. SalesTarget Financial Reconciliation

Aggregates actual sales revenue to the target grain (`Year`, `Month`, `Region`, `ProductCategory`), compares actual performance against sales targets, and measures overall variance.

In [3]:
sales_monthly = (
    sales
    .join(
        calendar.select("Date", "Year", "MonthNumber"),
        sales.OrderDate == calendar.Date,
        "left"
    )
    .groupBy(
        "Year",
        "MonthNumber",
        "Region",
        "Category"
    )
    .agg(
        spark_sum("LineAmount").alias("ActualSales")
    )
)

sales_monthly = sales_monthly.withColumnRenamed(
    "Category",
    "ProductCategory"
)

comparison = (
    sales_monthly
    .join(
        targets,
        (sales_monthly.Year == targets.Year) &
        (sales_monthly.MonthNumber == targets.Month) &
        (sales_monthly.Region == targets.Region) &
        (sales_monthly.ProductCategory == targets.ProductCategory),
        "left"
    )
    .withColumn(
        "Variance",
        col("ActualSales") - col("SalesTarget")
    )
)

display(comparison)

StatementMeta(, ee3da146-c5f9-430a-a980-1cf85008508b, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 624016d2-5711-4489-b829-aefd88ed915a)

In [4]:
from pyspark.sql.functions import col, sum as spark_sum, round

summary = (
    comparison
    .groupBy()
    .agg(
        round(spark_sum("ActualSales"), 2).alias("TotalActualSales"),
        round(spark_sum("SalesTarget"), 2).alias("TotalSalesTarget"),
        round(spark_sum("Variance"), 2).alias("TotalVariance")
    )
    .withColumn(
        "AchievementPercent",
        round(
            col("TotalActualSales") /
            col("TotalSalesTarget") * 100,
            2
        )
    )
)

display(summary)

StatementMeta(, ee3da146-c5f9-430a-a980-1cf85008508b, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 28585033-3897-4664-b462-dea93fa295ca)

### Target Coverage Verification

Verifies whether all 192 target combinations in `salestarget` have matching sales activity in `Sales`.

In [6]:
from pyspark.sql.functions import col

target_coverage = (
    targets.alias("t")
    .join(
        sales_monthly.alias("s"),
        (col("t.Year") == col("s.Year")) &
        (col("t.Month") == col("s.MonthNumber")) &
        (col("t.Region") == col("s.Region")) &
        (col("t.ProductCategory") == col("s.ProductCategory")),
        "left"
    )
    .select(
        col("t.Year"),
        col("t.Month"),
        col("t.Region"),
        col("t.ProductCategory"),
        col("t.SalesTarget"),
        col("s.ActualSales")
    )
)

print(
    "Total SalesTarget combinations:",
    target_coverage.count()
)

print(
    "Targets with no Sales:",
    target_coverage.filter(
        col("ActualSales").isNull()
    ).count()
)

display(
    target_coverage
    .filter(col("ActualSales").isNull())
)

StatementMeta(, ee3da146-c5f9-430a-a980-1cf85008508b, 8, Finished, Available, Finished, False)

Total SalesTarget combinations: 192
Targets with no Sales: 0


SynapseWidget(Synapse.DataFrame, e67c7347-7abf-4199-a403-58619281a1b9)

## 8. Marketing Campaign vs. Sales Category Reconciliation

Compares campaign revenue against actual sales revenue by product category. Campaign revenue is evaluated as an independent marketing performance metric rather than a direct line-item reconciliation.

In [8]:
from pyspark.sql.functions import col, sum as spark_sum, round

# Sales revenue by product category
sales_category = (
    spark.table("Sales")
    .withColumn("LineAmount", col("LineAmount").cast("double"))
    .groupBy("Category")
    .agg(
        spark_sum("LineAmount").alias("SalesRevenue")
    )
    .withColumnRenamed("Category", "ProductCategory")
)

# Marketing campaign revenue by product category
campaign_category = (
    spark.table("marketing_campaign")
    .withColumn(
        "RevenueGenerated",
        col("RevenueGenerated").cast("double")
    )
    .groupBy("ProductCategory")
    .agg(
        spark_sum("RevenueGenerated").alias("CampaignRevenue")
    )
)

# Compare the two
category_comparison = (
    campaign_category
    .join(
        sales_category,
        "ProductCategory",
        "full"
    )
    .fillna(0, ["CampaignRevenue", "SalesRevenue"])
    .withColumn(
        "Difference",
        col("CampaignRevenue") - col("SalesRevenue")
    )
    .withColumn(
        "CampaignToSalesRatio",
        round(
            col("CampaignRevenue") /
            col("SalesRevenue") * 100,
            2
        )
    )
)

display(
    category_comparison.orderBy("ProductCategory")
)

StatementMeta(, ee3da146-c5f9-430a-a980-1cf85008508b, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 78889e55-608f-4fb4-bcfc-2266241c7ab2)

## 9. Final Validation Summary

Consolidates Gold layer quality checks to confirm readiness for semantic modeling.

In [9]:
print("======================================")
print("       GOLD VALIDATION SUMMARY")
print("======================================")

print("\n1. KEY INTEGRITY")
print("All Gold table primary/business keys validated.")
print("Duplicate and null key violations: 0")

print("\n2. REFERENTIAL INTEGRITY")
print("Sales foreign-key relationships validated.")
print("Orphan records: 0")

print("\n3. SALESTARGET GRAIN")
print("Duplicate Year + Month + Region + ProductCategory: 0")

print("\n4. SALES BUSINESS VALIDATION")
print("Invalid Quantity: 0")
print("Invalid UnitPrice: 0")
print("Invalid CostAtSale: 0")
print("Invalid Discount: 0")
print("LineAmount mismatches: 0")

print("\n5. MARKETING CAMPAIGN BUSINESS VALIDATION")
print("Invalid Spend: 0")
print("Invalid Revenue: 0")
print("Invalid Impressions: 0")
print("Invalid Clicks: 0")
print("Invalid Conversions: 0")

print("\n6. SALES VS SALESTARGET")
print("All 192 target combinations have matching sales activity.")

print("\n7. MARKETING VS SALES")
print("Product-category relationships validated.")
print("Campaign revenue is treated as an independent marketing metric.")

print("\n======================================")
print("       GOLD VALIDATION COMPLETE")
print("======================================")

StatementMeta(, ee3da146-c5f9-430a-a980-1cf85008508b, 11, Finished, Available, Finished, False)

       GOLD VALIDATION SUMMARY

1. KEY INTEGRITY
All Gold table primary/business keys validated.
Duplicate and null key violations: 0

2. REFERENTIAL INTEGRITY
Sales foreign-key relationships validated.
Orphan records: 0

3. SALESTARGET GRAIN
Duplicate Year + Month + Region + ProductCategory: 0

4. SALES BUSINESS VALIDATION
Invalid Quantity: 0
Invalid UnitPrice: 0
Invalid CostAtSale: 0
Invalid Discount: 0
LineAmount mismatches: 0

5. MARKETING CAMPAIGN BUSINESS VALIDATION
Invalid Spend: 0
Invalid Revenue: 0
Invalid Impressions: 0
Invalid Clicks: 0
Invalid Conversions: 0

6. SALES VS SALESTARGET
All 192 target combinations have matching sales activity.

7. MARKETING VS SALES
Product-category relationships validated.
Campaign revenue is treated as an independent marketing metric.

       GOLD VALIDATION COMPLETE
